# Observables: Diagnosing Phases in Translation-Invariant ED

## Motivation and Scope

Symmetry-resolved exact diagonalisation gives us energy eigenvalues and eigenvectors labelled by symmetry quantum numbers.  The next question is: **what do we measure to identify the phase?**

For translation-invariant Hamiltonians with periodic boundary conditions, many naive diagnostics fail for fundamental, not numerical, reasons.  This notebook explains why, and provides the correct tools for distinguishing:

- **Fractional Chern insulators** (FCI)
- **Charge-density waves / Wigner crystals** (CDW)
- **Anomalous Hall crystals** (AHC)
- **Superfluids** (SF), including those condensing at specific crystal momenta
- **Supersolids** (SS)

## Why ⟨n_i⟩ is Useless for Translation-Invariant PBC Systems

### The fundamental problem

Consider a translation-invariant Hamiltonian on a finite lattice with periodic boundary conditions:

\begin{equation}
    [H, T_{\bm R}] = 0, \qquad \forall \text{ lattice translations } T_{\bm R}.
\end{equation}

A **non-degenerate** ground state $|\psi\rangle$ of any such Hamiltonian is necessarily a simultaneous eigenstate of all translations:

\begin{equation}
    T_{\bm R} |\psi\rangle = e^{i\bm k\cdot\bm R} |\psi\rangle.
\end{equation}

Consequently, the one-point density is **identically uniform**:

\begin{equation}
    \boxed{\langle n_i\rangle = \langle\psi|T_{-\bm R}\,n_i\,T_{\bm R}|\psi\rangle = \langle n_{i+\bm R}\rangle \;\Rightarrow\; \langle n_i\rangle \equiv \frac{N_e}{N}}.
\end{equation}

This is **not** a sign that the system is in a fluid phase.  It is a mathematical identity that holds for **any** translation-invariant eigenstate — CDW, FCI, superfluid, everything.  The CDW order is hidden by the quantum superposition of all translation-related patterns.

### Three flawed approaches (and why they fail)

**1. Single symmetry-sector ED.**  Whether you use `:identity` group (full real-space ED, all Fock masks) or a single translation sector `(k₁,k₂)`, the one-point density $\langle n_i\rangle$ is uniformly $N_e/N$.  For a single 1D irrep, the projected density operator averages over the translation orbit:

\begin{equation}
    \langle\chi|\hat n_i|\chi\rangle = \frac{1}{|G|}\sum_g |\chi(g)|^2 \langle n_{g(i)}\rangle = \frac{N_e}{N}.
\end{equation}

**2. Accidental ground-state degeneracy in full-ED.**  When LAPACK diagonalises a degenerate subspace (e.g., fermionic ν=2/3 FCI on [2,3] checkerboard), it may return an arbitrary linear combination of symmetry-distinct ground states.  The resulting "modulation" in $\langle n_i\rangle$ is a **basis-choice artifact** — change the sample size and the pattern disappears.  It does not reflect the physical CDW ordering wavevector $\bm Q$.

**3. Coherent superposition of nearly-degenerate sectors.**  One *can* construct $|\psi\rangle = \alpha|\psi_{\bm k}\rangle + \beta|\psi_{\bm k+\bm Q}\rangle$ and observe real-space modulation from the cross term $\langle\psi_{\bm k}|\hat n_i|\psi_{\bm k+\bm Q}\rangle$.  However, this requires (a) knowing $\bm Q$ in advance, (b) carefully choosing the relative phase, and (c) the result depends on these choices.  It is not a clean diagnostic.

**Conclusion:** For translation-invariant PBC systems, real-space $\langle n_i\rangle$ is meaningless.  The `vertices_occupation_distribution` single-sector API has been **removed**; `vertices_occupation_distribution_full_ed` now **requires OBC** (`@assert !any(pbc_indicator)`).

## The Correct Diagnostic Toolkit

### Overview

| Phase | Many-body Chern | $S(\bm q)$ (static structure factor) | Superfluid stiffness $\rho_s$ | $\rho(\bm k)$ (ODLRO) | Low-energy spectrum |
|-------|:---:|:---:|:---:|:---:|:---:|
| **FCI** | fractional (e.g. 1/2, 2/3) | smooth, no Bragg peaks | $\approx 0$ | broad, $O(1)$ | topological GSD (e.g. 2, 3) |
| **CDW / Wigner crystal** | 0 | **sharp Bragg peaks** at $\bm Q$ | $\approx 0$ | broad, $O(1)$ | tower of states (spacing $\bm Q$) |
| **Anomalous Hall crystal** | integer | sharp Bragg peaks at $\bm Q$ | $\approx 0$ | broad, $O(1)$ | topological GSD + tower |
| **SF @ k-point** | 0 | smooth, no Bragg peaks | **finite** | **macroscopic peak** at $\bm k^*$ | gapless (hard in finite size) |
| **Supersolid** | 0 | sharp Bragg peaks | **finite** | **macroscopic peak** at $\bm k^*$ | tower + gapless |

### 1. Static Structure Factor $S(\bm q)$ — diagnosing charge order

The connected density-density correlation function in momentum space:

\begin{equation}
    \boxed{S^{\alpha\beta}(\bm q) = \frac{1}{N}\sum_{i,j} e^{i\bm q\cdot(\bm r_i-\bm r_j)}\,\big(\langle n_i^\alpha n_j^\beta\rangle - \langle n_i^\alpha\rangle\langle n_j^\beta\rangle\big)}.
\end{equation}

$\alpha,\beta$ are optional flavour labels (e.g., sublattice A vs B).  The default `flavor_a = flavor_b = i -> true` gives the total density structure factor.

**Key properties:**
- Unlike $\langle n_i\rangle$, the **connected** correlator $\langle n_i n_j\rangle - \langle n_i\rangle\langle n_j\rangle$ carries the genuine two-point charge correlations even in a single translation sector.
- A CDW/Wigner crystal shows **sharp Bragg peaks** at the ordering wavevector $\bm Q$ and its harmonics.
- An FCI or superfluid shows a **smooth, featureless** $S(\bm q)$.

**Implementation:** `static_structure_factor(model, sector_label; ...)` computes $S(\bm q)$ at all allowed BZ momenta.  `compute_structure_factor_map(...)` scans a dense $\bm k$-grid in $[-1.5\pi, 1.5\pi]^2$ and returns data suitable for `CairoMakie.heatmap`.

**Relationship to magnetoroton physics:** In FQH/FCI systems, the magnetoroton mode softens at a finite wavevector $\bm q^*$ as the system approaches a CDW transition.  The static $S(\bm q)$ peaks at exactly this $\bm q^*$, and when the roton gap closes, $S(\bm q^*)$ diverges — this $\bm q^*$ **is** the CDW ordering wavevector $\bm Q$.

### 2. Off-Diagonal Long-Range Order $\rho(\bm k)$ — diagnosing superfluidity

The one-body density matrix and its Fourier transform:

\begin{equation}
    \boxed{\rho_{ij} = \langle a_i^\dagger a_j\rangle},\qquad
    \boxed{\rho(\bm k) = \frac{1}{N}\sum_{i,j} e^{i\bm k\cdot(\bm r_i-\bm r_j)}\,\langle a_i^\dagger a_j\rangle}.
\end{equation}

**Key properties:**
- The eigenvalues of $\rho_{ij}$ are the **natural orbital occupations**.  A superfluid has one (or a few) macroscopic eigenvalues $\sim O(N_e)$ — this is the Penrose-Onsager criterion for Bose condensation.
- $\rho(\bm k)$ directly shows **where** in the Brillouin zone the condensation occurs: a sharp peak at $\bm k^*$ with weight $\sim O(N_e)$.
- For an FCI or CDW insulator, $\rho(\bm k)$ is broad with all values $\sim O(1)$.
- For hard-core bosons, $\rho_{ij}$ is pure real-space: $a_i^\dagger a_j$ hops a particle from $j$ to $i$.  The ED implementation applies every valid hopping move and accumulates $\langle\psi|a_i^\dagger a_j|\psi\rangle$.

**Implementation:** `off_diagonal_long_range_order(model, sector_label; ...)` returns the full $N\times N$ complex matrix $\rho_{ij}$.  `compute_odlro_map(...)` performs the Fourier transform on a dense $\bm k$-grid.

**Symmetry-basis evaluation:** In a translation sector, a fixed local operator $a_i^\dagger a_j$ is not itself a symmetry-invariant Hamiltonian term.  Therefore one cannot apply it only to the orbit representative and reuse the Hamiltonian matrix-element shortcut.  The correct normalized orbit basis is
\begin{equation}
    |[s];\chi\rangle = \frac{1}{\sqrt{|G|\,|\mathrm{Stab}(s)|}}
    \sum_{g\in G}\chi(g)^*\,U_g|s\rangle .
\end{equation}
The implementation loops over every ket orbit member $U_g|s\rangle$, applies $a_i^\dagger a_j$ to that raw Fock configuration, canonicalizes the scattered mask, and obtains the coherent bra coefficient from `project_to_sector`.  This is algebraically identical to expanding the eigenvector back to the full real-space Fock basis, but it avoids materializing the full amplitude dictionary.

**Performance caution:** This exact orbit-basis contraction removes the dominant memory bottleneck of full-Fock expansion.  It does not reduce the formal all-to-all one-body workload: constructing the full $\rho_{ij}$ still scales like $N_{\mathrm{orbit}}\,|G|\,N_e\,N$ for hard-core particles.  For large clusters this is much more memory-stable, but it can still be CPU-heavy.

**Important:** $\rho(\bm k)$ and the Fourier transform of $\langle n_i\rangle$ are **completely different objects**.  $\rho(\bm k)$ measures off-diagonal (phase) coherence; $\mathcal{F}[\langle n_i\rangle]$ measures diagonal (density) modulation.  Only $\rho(\bm k)$ can detect superfluidity.

### 3. Many-Body Chern Number — diagnosing topology

Measured via the **charge pump** (see `doc/charge_pump.ipynb` and `test_bosonic_fci_charge_pump`):

\begin{equation}
    C = \frac{1}{2\pi}\int_0^{2\pi} d\theta\;\partial_\theta\,P(\theta),
    \qquad
    P(\theta) = \frac{1}{2\pi}\Im\ln\langle\Psi(\theta)|e^{i\frac{2\pi}{L}\hat X}|\Psi(\theta)\rangle.
\end{equation}

Over one flux quantum $\theta: 0\to 2\pi$, the pumped charge $\Delta P$ equals the many-body Chern number:
- FCI: fractional $C$ (e.g., $\pm 1/2$, $\pm 2/3$)
- CDW / SF: $C = 0$
- Anomalous Hall crystal: integer $C$

### 4. Superfluid Stiffness (not yet implemented)

The superfluid stiffness measures the energy cost of a phase twist:

\begin{equation}
    \rho_s = \left.\frac{\partial^2 E_0(\bm\theta)}{\partial\theta_\mu^2}\right|_{\bm\theta=0}.
\end{equation}

- Finite $\rho_s$ signals superfluidity (or supersolid).
- $\rho_s \approx 0$ signals an insulator (FCI or CDW).

This can be implemented by extending the existing flux-insertion infrastructure in `spectrum_flow.jl`.

### 5. Low-Energy Spectrum — tower of states vs topological degeneracy

The pattern of nearly-degenerate low-lying energy levels differs between phases:

- **FCI:** Topological ground-state degeneracy (GSD) — e.g. 2 states for bosonic ν=1/2 Laughlin, 3 for fermionic ν=2/3.  The GSD states appear at **specific** crystal momenta determined by the anyon statistics and the torus geometry.

- **CDW / Wigner crystal:** "Tower of states" — $d$ nearly-degenerate levels (where $d$ is the number of classically degenerate CDW patterns, i.e., the period) that collapse to exact degeneracy in the thermodynamic limit.  These appear at momentum sectors spaced by the ordering wavevector $\bm Q$:
  \begin{equation}
      \bm k_0, \; \bm k_0 + \bm Q, \; \bm k_0 + 2\bm Q, \; \ldots, \; \bm k_0 + (d-1)\bm Q.
  \end{equation}

- **SF:** No robust gap — the system is gapless in the thermodynamic limit.  Finite-size gaps are spurious and scale as $1/L$.

The momentum spacing between nearly-degenerate levels is itself a diagnostic: the spacing $\Delta\bm k = \bm Q$ is the ordering wavevector.

## ⚠️  Practical Cautions

### 1. $S(\bm q)$ vs $I(q) = |\sum_i e^{i\bm q\cdot\bm r_i}\langle n_i\rangle|^2$

The trivial Fourier transform of $\langle n_i\rangle$ is **not** a substitute for $S(\bm q)$.  Since $\langle n_i\rangle$ is uniform, $I(\bm q) = \delta_{\bm q,0}\,N_e^2/N$ — only the $\bm q=0$ component is non-zero.  This contains **no information** about charge order.

### 2. $S(\bm q)$ vs $n(\bm k)$ (momentum distribution)

$S(\bm q)$ measures **diagonal** (density-density) correlations.  $n(\bm k)$ — the Fourier transform of $\langle a_i^\dagger a_j\rangle$ — measures **off-diagonal** (single-particle) correlations.  They diagnose different physics:
- $S(\bm q)$: charge order (CDW $\leftrightarrow$ FCI distinction)
- $n(\bm k) \equiv \rho(\bm k)$: superfluidity (SF $\leftrightarrow$ insulator distinction)

### 3. Choice of symmetry sector

Both $S(\bm q)$ and $\rho(\bm k)$ are well-defined within a **single** translation sector.  $\rho(\bm k)$ may show the condensation peak directly.  For $S(\bm q)$, the connected correlator $\langle n_i n_j\rangle_c$ is non-trivial even in a single sector — the CDW information is encoded in **two-point** correlations, not one-point.

### 4. Disc geometry

On a disc with open boundaries, spontaneous translational symmetry breaking **can** be observed directly in $\langle n_i\rangle$.  This is the only setting where `vertices_occupation_distribution_full_ed` is physically meaningful.  However, disc ED has strong edge effects and is limited to very small system sizes.

## API Reference

### Static Structure Factor

```julia
# At discrete BZ momenta
qs, S_q = static_structure_factor(model, sector_label;
    target_eigval_idx = 1,
    filling_fraction,
    flavor_a = i -> true,
    flavor_b = i -> true,
    ed_mode = :matrix,
    ed_data = nothing,
)

# Dense k-grid heatmap
kx, ky, S_map = compute_structure_factor_map(model, sector_label;
    target_eigval_idx = 1,
    filling_fraction,
    flavor_a = i -> true,
    flavor_b = i -> true,
    k_resolution = 61,
    ed_data = nothing,
)
heatmap(kx, ky, S_map)  # CairoMakie
```

### Off-Diagonal Long-Range Order

```julia
# Full N×N one-body density matrix
ρ = off_diagonal_long_range_order(model, sector_label;
    target_eigval_idx = 1,
    filling_fraction,
    ed_mode = :matrix,
    ed_data = nothing,
)

# Dense k-grid heatmap
kx, ky, ρ_map = compute_odlro_map(model, sector_label;
    target_eigval_idx = 1,
    filling_fraction,
    k_resolution = 61,
    ed_data = nothing,
)
heatmap(kx, ky, ρ_map)  # CairoMakie
```

### Occupation Distribution (OBC only)

```julia
⟨n⟩ = vertices_occupation_distribution_full_ed(model;
    target_eigval_idx = 1,
    filling_fraction,
    flavor_filter = i -> true,
    ed_mode = :matrix,
    ed_data = nothing,
)  # ⚠️ requires open boundary conditions
```

## Summary of Key Formulas

| Quantity | Formula |
|---|---|
| One-point density | $\langle n_i\rangle$ (uniform for translation-invariant PBC) |
| Connected two-point correlator | $\langle n_i n_j\rangle_c = \langle n_i n_j\rangle - \langle n_i\rangle\langle n_j\rangle$ |
| Static structure factor | $S(\bm q) = \frac{1}{N}\sum_{i,j} e^{i\bm q\cdot(\bm r_i-\bm r_j)}\,\langle n_i n_j\rangle_c$ |
| One-body density matrix | $\rho_{ij} = \langle a_i^\dagger a_j\rangle$ |
| Momentum distribution | $\rho(\bm k) = \frac{1}{N}\sum_{i,j} e^{i\bm k\cdot(\bm r_i-\bm r_j)}\,\langle a_i^\dagger a_j\rangle$ |
| Many-body Chern number | $C = \frac{1}{2\pi}\int_0^{2\pi} d\theta\;\partial_\theta P(\theta)$, via charge pump |
| Superfluid stiffness | $\rho_s = \partial^2 E_0/\partial\theta^2\|_{\theta=0}$ |

The implementation files are:
- `observables/density_distribution.jl` — OBC-only real-space $\langle n_i\rangle$
- `observables/static_structure_factor.jl` — $S(\bm q)$ with BZ heatmap
- `observables/off_diagonal_long_range_order.jl` — $\rho_{ij}$ and $\rho(\bm k)$
- `observables/charge_pump.jl` — many-body Chern number via flux insertion
- `observables/spectrum_flow.jl` — spectral flow diagnostics